# Tyre RUL Data Cleaning Notebook

## Future TODO (Roadmap for Next Iterations)
1. Try baseline ML models first: Linear Regression, Random Forest, XGBoost, LightGBM, and CatBoost.
2. Try deep learning models for tabular data: MLP, TabTransformer, and FT-Transformer.
3. Compare CPU vs GPU training speed (especially XGBoost/LightGBM with CUDA when available).
4. Add experiment tracking (for example: MLflow or Weights & Biases) to compare model versions.
5. Build an LLM interface that converts natural-language driving descriptions into model-ready features.
6. Add a vehicle-spec enrichment step for unseen cars (fetch curb weight, power, etc. from trusted sources).
7. Wrap the final model in a prediction API (FastAPI) for frontend integration.
8. Add uncertainty estimates so the app can show confidence with each prediction.


## 1) Setup and Imports

In this section, we load the Python libraries we need and set notebook display options so tables are easier to read.


In [1]:
# Import a helper to work with file paths.
from pathlib import Path

# Import core numerical and table libraries.
import numpy as np
import pandas as pd

# Import display helper for nicer table output in notebooks.
from IPython.display import display

# Make pandas tables easier to read during debugging.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


## 2) Load the Dataset

Here we load the CSV file and do a quick shape check so we know the data was read correctly.


In [2]:
# Set the dataset path (same folder as this notebook).
DATA_PATH = Path("Synthetic Automobile Tyre RUL Data.csv")

# Stop with a clear message if the file is missing.
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH.resolve()}")

# Load the raw dataset into memory.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

# Print a quick shape summary.
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

# Show a small preview.
display(df_raw.head(5))


Rows: 968,143
Columns: 31


,vehicle_model,fuel_type,transmission_type,maximum_power(hp),maximum_torque(N/m),maximum_speed (km/h),steering_radius(m),vehicle_acceleration(0-100 km/h in seconds),vehicle_mileage(mpg),vehicle_sprung_mass(kg),tyre_camber_angle(degree),tyre_brand,tyre_size,tread_material,Standard_tread_depth(mm),tread_pattern,country,tread_wear_rating (UTQG),average_tread_temperature(celsius),recommended_inflation_pressure(psi),average_inflation_pressure(psi),tyre_age(years),number_of_punctures,current_tread_depth(mm),road_condition,weather_condition,axle_type(driven/dead),expected_tyre_life(km),retreaded,kilometers_driven(km),remaining_useful_life(km)
0,Mazda CX-5,petrol,manual,140,320,200,5.6,3.3,26,1650,-1.0,Toyo A23,225/65R17,Silica compound,7.14,Asymmetric,Brazil,300,22,35,22.10,1,4,5.94,smooth,Tropical and humid,dead,60000,no,7500,32931
1,BMW 3 Series,petrol,automatic,190,400,250,5.6,5.0,30,1575,-1.0,Bridgestone Turanza T001,225/45R18,Silica compound,6.35,Asymmetric,Japan,320,15,32,28.83,5,11,1.56,smooth,Mild with distinct seasons,drive,80000,no,57143,15819
2,Hyundai Elantra,diesel,manual,110,265,205,5.4,3.1,33,1350,-1.0,Kumho Solus TA31,205/55R16,silica and carbon black compound,7.14,Asymmetric,Brazil,500,22,35,29.72,2,3,6.04,smooth,Tropical and humid,drive,95000,no,13571,66767
3,Tesla Model S,electric,automatic,500,854,322,5.8,11.6,396,2241,-1.0,Michelin Pilot Sport 4S,245/45R19,silica and carbon black compound,7.14,Asymmetric,Canada,300,-5,36,25.27,3,3,3.64,offroad,Cold,drive,50000,no,21429,19837
4,Subaru Outback,petrol,manual,194,376,210,5.4,3.7,26,1600,-1.0,Yokohama Geolandar G91F,225/65R17,synthetic rubber,7.94,Asymmetric,United Kingdom,300,10,51,33.46,7,7,1.73,rough,Mixed conditions,dead,80000,no,43077,10198


## 3) Quick Data Inspection

Before cleaning, we check column names, data types, and missing values. This helps us catch issues early.


In [3]:
# Show all column names so we can match exact spelling from the CSV.
print("Columns in dataset:")
for col in df_raw.columns:
    print(f"- {col}")

# Build a small table of missing-value percentages.
missing_pct = (df_raw.isna().mean() * 100).sort_values(ascending=False).round(2)
missing_table = missing_pct.reset_index()
missing_table.columns = ["column", "missing_pct"]

# Display dtypes and top missing columns.
display(df_raw.dtypes.to_frame(name="dtype").head(31))
display(missing_table.head(15))


Columns in dataset:
- vehicle_model
- fuel_type
- transmission_type
- maximum_power(hp)
- maximum_torque(N/m)
- maximum_speed (km/h)
- steering_radius(m)
- vehicle_acceleration(0-100 km/h in seconds)
- vehicle_mileage(mpg)
- vehicle_sprung_mass(kg)
- tyre_camber_angle(degree)
- tyre_brand
- tyre_size
- tread_material
- Standard_tread_depth(mm)
- tread_pattern
- country
- tread_wear_rating (UTQG)
- average_tread_temperature(celsius)
- recommended_inflation_pressure(psi)
- average_inflation_pressure(psi)
- tyre_age(years)
- number_of_punctures
- current_tread_depth(mm)
- road_condition
- weather_condition
- axle_type(driven/dead)
- expected_tyre_life(km)
- retreaded
- kilometers_driven(km)
- remaining_useful_life(km)


,dtype
vehicle_model,object
fuel_type,object
transmission_type,object
maximum_power(hp),int64
maximum_torque(N/m),int64
maximum_speed (km/h),int64
steering_radius(m),float64
vehicle_acceleration(0-100 km/h in seconds),float64
vehicle_mileage(mpg),int64
vehicle_sprung_mass(kg),int64


,column,missing_pct
0,vehicle_model,0.0
1,fuel_type,0.0
2,transmission_type,0.0
3,maximum_power(hp),0.0
4,maximum_torque(N/m),0.0
5,maximum_speed (km/h),0.0
6,steering_radius(m),0.0
7,vehicle_acceleration(0-100 km/h in seconds),0.0
8,vehicle_mileage(mpg),0.0
9,vehicle_sprung_mass(kg),0.0


## 4) Pick Features for Modeling

We keep your core tire-focused features, then add a few extra vehicle and usage features that usually help tire-life prediction.


In [4]:
# Define the prediction target.
target_col = "remaining_useful_life(km)"

# Keep your proposed core tire-life features.
core_features = [
    "current_tread_depth(mm)",
    "Standard_tread_depth(mm)",
    "kilometers_driven(km)",
    "tread_wear_rating (UTQG)",
    "average_inflation_pressure(psi)",
    "recommended_inflation_pressure(psi)",
]

# Add extra context features that can improve prediction quality.
additional_features = [
    "vehicle_sprung_mass(kg)",
    "vehicle_acceleration(0-100 km/h in seconds)",
    "maximum_power(hp)",
    "maximum_torque(N/m)",
    "maximum_speed (km/h)",
    "vehicle_mileage(mpg)",
    "average_tread_temperature(celsius)",
    "tyre_age(years)",
    "number_of_punctures",
    "expected_tyre_life(km)",
    "retreaded",
    "road_condition",
    "weather_condition",
    "axle_type(driven/dead)",
    "vehicle_model",
    "fuel_type",
    "transmission_type",
    "tyre_brand",
    "tyre_size",
    "tread_material",
    "country",
]

# Merge lists while preserving order and removing duplicates.
selected_features = list(dict.fromkeys(core_features + additional_features))

# Confirm that every selected column exists in the dataset.
missing_cols = [c for c in selected_features + [target_col] if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f"These columns were not found: {missing_cols}")

# Create a modeling dataframe with selected features + target.
df_model = df_raw[selected_features + [target_col]].copy()

# Print a small summary of selected columns.
print(f"Selected features: {len(selected_features)}")
print(f"Target: {target_col}")
display(df_model.head(3))


Selected features: 27
Target: remaining_useful_life(km)


,current_tread_depth(mm),Standard_tread_depth(mm),kilometers_driven(km),tread_wear_rating (UTQG),average_inflation_pressure(psi),recommended_inflation_pressure(psi),vehicle_sprung_mass(kg),vehicle_acceleration(0-100 km/h in seconds),maximum_power(hp),maximum_torque(N/m),maximum_speed (km/h),vehicle_mileage(mpg),average_tread_temperature(celsius),tyre_age(years),number_of_punctures,expected_tyre_life(km),retreaded,road_condition,weather_condition,axle_type(driven/dead),vehicle_model,fuel_type,transmission_type,tyre_brand,tyre_size,tread_material,country,remaining_useful_life(km)
0,5.94,7.14,7500,300,22.10,35,1650,3.3,140,320,200,26,22,1,4,60000,no,smooth,Tropical and humid,dead,Mazda CX-5,petrol,manual,Toyo A23,225/65R17,Silica compound,Brazil,32931
1,1.56,6.35,57143,320,28.83,32,1575,5.0,190,400,250,30,15,5,11,80000,no,smooth,Mild with distinct seasons,drive,BMW 3 Series,petrol,automatic,Bridgestone Turanza T001,225/45R18,Silica compound,Japan,15819
2,6.04,7.14,13571,500,29.72,35,1350,3.1,110,265,205,33,22,2,3,95000,no,smooth,Tropical and humid,drive,Hyundai Elantra,diesel,manual,Kumho Solus TA31,205/55R16,silica and carbon black compound,Brazil,66767


## 5) Feature Engineering (Helpful Derived Columns)

This step creates a few extra features from your core variables, like wear percentage and pressure gap, to give the model more signal.


In [5]:
# Create safe denominators to avoid divide-by-zero issues.
safe_standard_depth = df_model["Standard_tread_depth(mm)"].replace(0, np.nan)
safe_recommended_pressure = df_model["recommended_inflation_pressure(psi)"].replace(0, np.nan)
safe_expected_life = df_model["expected_tyre_life(km)"].replace(0, np.nan)

# Measure how much tread has already been used (in mm).
df_model["tread_depth_used_mm"] = df_model["Standard_tread_depth(mm)"] - df_model["current_tread_depth(mm)"]

# Convert tread usage into percentage of the original tread.
df_model["tread_depth_used_pct"] = df_model["tread_depth_used_mm"] / safe_standard_depth

# Compute pressure difference between real usage and recommended value.
df_model["pressure_gap_psi"] = df_model["average_inflation_pressure(psi)"] - df_model["recommended_inflation_pressure(psi)"]

# Convert pressure gap into percentage to normalize across tire setups.
df_model["pressure_gap_pct"] = df_model["pressure_gap_psi"] / safe_recommended_pressure

# Show how much of expected life has already been used.
df_model["km_used_ratio_vs_expected"] = df_model["kilometers_driven(km)"] / safe_expected_life

# Preview engineered columns.
display(
    df_model[
        [
            "tread_depth_used_mm",
            "tread_depth_used_pct",
            "pressure_gap_psi",
            "pressure_gap_pct",
            "km_used_ratio_vs_expected",
        ]
    ].head(5)
)


,tread_depth_used_mm,tread_depth_used_pct,pressure_gap_psi,pressure_gap_pct,km_used_ratio_vs_expected
0,1.20,0.168067,-12.90,-0.368571,0.125000
1,4.79,0.754331,-3.17,-0.099063,0.714287
2,1.10,0.154062,-5.28,-0.150857,0.142853
3,3.50,0.490196,-10.73,-0.298056,0.428580
4,6.21,0.782116,-17.54,-0.343922,0.538462


## 6) Data Cleaning Pipeline

Now we clean the selected data: standardize text, handle invalid values, fill missing values, and clip extreme outliers.


In [6]:
# Make a working copy so the raw table remains unchanged.
df_clean = df_model.copy()

# Find text columns before numeric conversion.
categorical_cols = df_clean.select_dtypes(include=["object", "string"]).columns.tolist()

# Clean text columns by trimming spaces and normalizing case.
for col in categorical_cols:
    # Convert to pandas string dtype for safer text operations.
    df_clean[col] = df_clean[col].astype("string").str.strip()
    # Normalize values so categories are consistent.
    df_clean[col] = df_clean[col].str.lower()
    # Convert empty-like values into missing values.
    df_clean[col] = df_clean[col].replace({"": pd.NA, "nan": pd.NA, "none": pd.NA})

# Convert retreaded yes/no to 1/0 for model usage.
if "retreaded" in df_clean.columns:
    df_clean["retreaded"] = df_clean["retreaded"].map({"yes": 1, "no": 0})

# Convert non-text columns to numeric when possible.
for col in df_clean.columns:
    # Keep string columns as categories.
    if pd.api.types.is_string_dtype(df_clean[col]):
        continue
    # Coerce unexpected values into NaN.
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Replace any infinite values with NaN.
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Define basic rule-based bounds for data sanity.
rules = {
    "current_tread_depth(mm)": (0, None),
    "Standard_tread_depth(mm)": (0, None),
    "kilometers_driven(km)": (0, None),
    "tread_wear_rating (UTQG)": (0, None),
    "average_inflation_pressure(psi)": (0, None),
    "recommended_inflation_pressure(psi)": (0, None),
    "vehicle_sprung_mass(kg)": (0, None),
    "vehicle_acceleration(0-100 km/h in seconds)": (0, None),
    "maximum_power(hp)": (0, None),
    "maximum_torque(N/m)": (0, None),
    "maximum_speed (km/h)": (0, None),
    "vehicle_mileage(mpg)": (0, None),
    "average_tread_temperature(celsius)": (-80, 120),
    "tyre_age(years)": (0, None),
    "number_of_punctures": (0, None),
    "expected_tyre_life(km)": (0, None),
    "remaining_useful_life(km)": (0, None),
    "tread_depth_used_mm": (0, None),
    "km_used_ratio_vs_expected": (0, None),
}

# Apply bounds and set invalid values to NaN.
for col, (lower, upper) in rules.items():
    if col not in df_clean.columns:
        continue
    if lower is not None:
        df_clean.loc[df_clean[col] < lower, col] = np.nan
    if upper is not None:
        df_clean.loc[df_clean[col] > upper, col] = np.nan

# If current tread is above standard tread, mark current tread as invalid.
depth_issue = df_clean["current_tread_depth(mm)"] > df_clean["Standard_tread_depth(mm)"]
df_clean.loc[depth_issue, "current_tread_depth(mm)"] = np.nan

# Recompute depth-based derived columns after cleaning.
safe_standard_depth_clean = df_clean["Standard_tread_depth(mm)"].replace(0, np.nan)
df_clean["tread_depth_used_mm"] = df_clean["Standard_tread_depth(mm)"] - df_clean["current_tread_depth(mm)"]
df_clean["tread_depth_used_pct"] = df_clean["tread_depth_used_mm"] / safe_standard_depth_clean

# Separate numeric and categorical columns for imputation.
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df_clean.columns if c not in numeric_cols]

# Fill numeric NaN values using median (robust to outliers).
for col in numeric_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)

# Fill categorical NaN values using mode (most frequent label).
for col in categorical_cols:
    mode_values = df_clean[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "unknown"
    df_clean[col] = df_clean[col].fillna(fill_value)

# Clip extreme numeric feature values to reduce outlier impact.
for col in numeric_cols:
    if col == target_col:
        continue
    q1 = df_clean[col].quantile(0.01)
    q99 = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(q1, q99)

# Print quick quality checks.
print("Any missing values left:", bool(df_clean.isna().sum().sum()))
print("Cleaned shape:", df_clean.shape)
display(df_clean.head(5))


Any missing values left: False
Cleaned shape: (968143, 33)


,current_tread_depth(mm),Standard_tread_depth(mm),kilometers_driven(km),tread_wear_rating (UTQG),average_inflation_pressure(psi),recommended_inflation_pressure(psi),vehicle_sprung_mass(kg),vehicle_acceleration(0-100 km/h in seconds),maximum_power(hp),maximum_torque(N/m),maximum_speed (km/h),vehicle_mileage(mpg),average_tread_temperature(celsius),tyre_age(years),number_of_punctures,expected_tyre_life(km),retreaded,road_condition,weather_condition,axle_type(driven/dead),vehicle_model,fuel_type,transmission_type,tyre_brand,tyre_size,tread_material,country,remaining_useful_life(km),tread_depth_used_mm,tread_depth_used_pct,pressure_gap_psi,pressure_gap_pct,km_used_ratio_vs_expected
0,5.94,7.14,7500.0,300.0,22.10,35.0,1650.0,3.3,140.0,320.0,200.0,26.0,22.0,1.0,4.0,60000.0,0,smooth,tropical and humid,dead,mazda cx-5,petrol,manual,toyo a23,225/65r17,silica compound,brazil,32931.0,1.20,0.168067,-12.90,-0.368571,0.125000
1,1.56,6.35,57143.0,320.0,28.83,32.0,1575.0,5.0,190.0,400.0,250.0,30.0,15.0,5.0,11.0,80000.0,0,smooth,mild with distinct seasons,drive,bmw 3 series,petrol,automatic,bridgestone turanza t001,225/45r18,silica compound,japan,15819.0,4.79,0.754331,-3.17,-0.099063,0.714287
2,6.04,7.14,13571.0,500.0,29.72,35.0,1350.0,3.1,110.0,265.0,205.0,33.0,22.0,2.0,3.0,95000.0,0,smooth,tropical and humid,drive,hyundai elantra,diesel,manual,kumho solus ta31,205/55r16,silica and carbon black compound,brazil,66767.0,1.10,0.154062,-5.28,-0.150857,0.142853
3,3.64,7.14,21429.0,300.0,25.27,36.0,2241.0,11.6,500.0,854.0,322.0,396.0,-5.0,3.0,3.0,50000.0,0,offroad,cold,drive,tesla model s,electric,automatic,michelin pilot sport 4s,245/45r19,silica and carbon black compound,canada,19837.0,3.50,0.490196,-10.73,-0.298056,0.428580
4,1.73,7.94,43077.0,300.0,33.46,51.0,1600.0,3.7,194.0,376.0,210.0,26.0,10.0,7.0,7.0,80000.0,0,rough,mixed conditions,dead,subaru outback,petrol,manual,yokohama geolandar g91f,225/65r17,synthetic rubber,united kingdom,10198.0,6.21,0.782116,-17.54,-0.343922,0.538462


## 7) Save Cleaned Data for Next Steps

Finally, we save the cleaned data so your training notebook can directly load it without repeating all preprocessing steps.


In [7]:
# Create an output folder for generated artifacts.
output_dir = Path("artifacts")
output_dir.mkdir(exist_ok=True)

# Define output file paths.
clean_csv_path = output_dir / "tyre_rul_cleaned.csv"
clean_parquet_path = output_dir / "tyre_rul_cleaned.parquet"

# Save cleaned data in CSV format.
df_clean.to_csv(clean_csv_path, index=False)

# Save cleaned data in Parquet format.
df_clean.to_parquet(clean_parquet_path, index=False)

# Print where files were saved.
print(f"Saved CSV: {clean_csv_path.resolve()}")
print(f"Saved Parquet: {clean_parquet_path.resolve()}")

# Show remaining missing percentages after cleaning.
post_clean_missing = (df_clean.isna().mean() * 100).round(4).sort_values(ascending=False)
display(post_clean_missing.head(10))


Saved CSV: C:\GitHub_Repos\team_Tyre\artifacts\tyre_rul_cleaned.csv
Saved Parquet: C:\GitHub_Repos\team_Tyre\artifacts\tyre_rul_cleaned.parquet


current_tread_depth(mm)                        0.0
Standard_tread_depth(mm)                       0.0
kilometers_driven(km)                          0.0
tread_wear_rating (UTQG)                       0.0
average_inflation_pressure(psi)                0.0
recommended_inflation_pressure(psi)            0.0
vehicle_sprung_mass(kg)                        0.0
vehicle_acceleration(0-100 km/h in seconds)    0.0
maximum_power(hp)                              0.0
maximum_torque(N/m)                            0.0
dtype: float64

## 8) Optional: Quick CUDA Readiness Note

Data cleaning itself is mostly CPU work, but this quick check confirms your XGBoost package is ready for future modeling experiments.


In [8]:
# Try importing XGBoost for future model training.
try:
    # Import xgboost package.
    import xgboost as xgb
    # Print installed version.
    print("XGBoost version:", xgb.__version__)
    # Show a simple GPU training tip for later.
    print("For GPU training later, use parameters like: device='cuda', tree_method='hist'.")
except Exception as e:
    # Print message if package is not available yet.
    print("XGBoost is not ready yet:", e)


XGBoost version: 3.2.0
For GPU training later, use parameters like: device='cuda', tree_method='hist'.
